# Neuron set examples

Examples for how to use different types of neuron sets.

In [ ]:
import obi_one as obi
from pathlib import Path

### __Initialization:__ Loading a circuit

In [ ]:
circuit_name = "N_10__top_nodes_dim6"

circuit_path_prefix = Path("../../../../data/tiny_circuits")
matrix_path_prefix = Path("../../../../data/connectivity_matrices")  # OPTIONAL: Connectivity matrix path only required for some of the node sets in this example notebook; can be set to None

circuit_path = circuit_path_prefix / circuit_name / "circuit_config.json"
if matrix_path_prefix is None:
    matrix_path = None
else:
    matrix_path = matrix_path_prefix / circuit_name / "connectivity_matrix.h5"
circuit = obi.Circuit(name=circuit_name, path=str(circuit_path), matrix_path=str(matrix_path))

print(f"Circuit '{circuit}' with {circuit.sonata_circuit.nodes[circuit.default_population_name].size} neurons and {circuit.sonata_circuit.edges[circuit.default_edge_population_name].size} synapses")
print(f"Node populations: '{circuit.sonata_circuit.nodes.population_names}'")
print(f"Edge populations: '{circuit.sonata_circuit.edges.population_names}'")

### **Neuron set examples**

The neuron sets have `population` as a field.
They return:
- `dict[str, list[int]]` from `get_neuron_ids`: Neuron IDs by population
- `tuple[dict|list, dict]` from `get_node_set_definition`: Base expression + additional definitions in case of a compound expression

In [ ]:
from obi_one.scientific.blocks.neuron_sets.id import BiophysicalPopulationIDNeuronSet
from obi_one.scientific.blocks.neuron_sets.population import BiophysicalPopulationNeuronSet
from obi_one.scientific.blocks.neuron_sets.predefined import BiophysicalPopulationPredefinedNeuronSet, PredefinedNeuronSet, VirtualPopulationPredefinedNeuronSet
from obi_one.scientific.blocks.neuron_sets.property import BiophysicalPopulationPropertyNeuronSet, NeuronPropertyFilter
from obi_one.scientific.blocks.neuron_sets.specific import AllPopulationNeurons, AllBiophysicalNeurons
from obi_one.core.tuple import NamedTuple

#### (a) BiophysicalPopulationNeuronSet — sample % from a population

In [ ]:
nset = BiophysicalPopulationNeuronSet(population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=1)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PopulationNeuronSet (50%)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

#### (b) BiophysicalPopulationNeuronSet — full population (symbolic expression)

In [ ]:
nset = BiophysicalPopulationNeuronSet(population="S1nonbarrel_neurons")

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"BiophysicalPopulationNeuronSet (100%)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition (symbolic): {nset_def}")

#### (c) PredefinedNeuronSet — use an existing node set (multi-population)

In [ ]:
nset = PredefinedNeuronSet(node_set="AllPopulations")
nset.set_block_name("nset")

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PredefinedNeuronSet '{nset.node_set}'")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition (symbolic): {nset_def} {combined}")

# Force resolve IDs
nset_def_resolved, combined_resolved = nset.get_node_set_definition(circuit, force_resolve_ids=True)
print(f"  Node set definition (resolved): {nset_def_resolved} {combined_resolved}")

#### (d) BiophysicalPopulationPredefinedNeuronSet — predefined node set resolved in one population (with sampling)

In [ ]:
nset = BiophysicalPopulationPredefinedNeuronSet(
    node_set="Layer6", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=42
)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PredefinedPopulationNeuronSet 'Layer6' (50% sample)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

In [ ]:
# Wrong population type
nset = VirtualPopulationPredefinedNeuronSet(
    node_set="Layer6", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=42
)

# This should raise a ValueError about wrong population type
try:
    nset_def, combined = nset.get_node_set_definition(circuit)
except ValueError as e:
    print(f"Caught expected error: {e}")

#### (e) BiophysicalPopulationIDNeuronSet — by neuron IDs

In [ ]:
nset = BiophysicalPopulationIDNeuronSet(
    population="S1nonbarrel_neurons",
    neuron_ids=NamedTuple(name="my_ids", elements=(0, 2, 5, 8)),
)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"BiophysicalPopulationIDNeuronSet")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

#### (f) BiophysicalPopulationPropertyNeuronSet — by neuron properties

In [ ]:
nset = BiophysicalPopulationPropertyNeuronSet(
    population="S1nonbarrel_neurons",
    property_filter=NeuronPropertyFilter(filter_dict={"layer": ["6"], "synapse_class": ["EXC"]}),
)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PropertyNeuronSet (layer=6, synapse_class=EXC)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

# With sampling
nset_sampled = BiophysicalPopulationPropertyNeuronSet(
    population="S1nonbarrel_neurons",
    property_filter=NeuronPropertyFilter(filter_dict={"layer": ["6"], "synapse_class": ["EXC"]}),
    sample_percentage=50,
    sample_seed=7,
)
nset_sampled.set_block_name("prop_nset_sampled")
ids_sampled = nset_sampled.get_neuron_ids(circuit)
print(f"\n  With 50% sampling: {ids_sampled}")

#### (g) AllPopulationNeurons — select all neurons from all populations (specific neuron sets)

In [ ]:
# AllNeurons — all neurons from all populations
nset = AllPopulationNeurons()
nset.set_block_name("all_neurons")

ids = nset.get_neuron_ids(circuit)
print(f"AllNeurons")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")

# Symbolic node set definition
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"  Node set definition (symbolic): {nset_def}")
if combined:
    print(f"  Combined definitions: {combined}")

# Force resolve IDs
nset_def_resolved, combined_resolved = nset.get_node_set_definition(circuit, force_resolve_ids=True)
print(f"  Node set definition (resolved): {nset_def_resolved}")
if combined_resolved:
    print(f"  Combined definitions (resolved): {combined_resolved}")

In [ ]:
# AllBiophysicalNeurons — only biophysical populations
nset_bio = AllBiophysicalNeurons()
nset_bio.set_block_name("all_biophysical")

ids_bio = nset_bio.get_neuron_ids(circuit)
nset_def_bio, _ = nset_bio.get_node_set_definition(circuit)
print(f"AllBiophysicalNeurons")
print(f"  Populations: {nset_bio.get_populations(circuit)}")
print(f"  Neuron IDs: {ids_bio}")
print(f"  Node set definition (symbolic): {nset_def_bio}")

#### (h) Writing a node set to file

In [ ]:
import json

nset = PredefinedNeuronSet(node_set="Layer6")
nset.set_block_name("my_layer6_nset")

output_file = nset.to_node_set_file(
    circuit, output_path="./", file_name="neuron_set_test.json",
    overwrite_if_exists=True, init_empty=True
)
print(f"Written to: {output_file}")

with open(output_file) as f:
    print(json.dumps(json.load(f), indent=2))

# With force_resolve_ids=True
output_file_resolved = nset.to_node_set_file(
    circuit, output_path="./", file_name="neuron_set_test_resolved.json",
    overwrite_if_exists=True, init_empty=True, force_resolve_ids=True
)
print(f"\nWritten (resolved) to: {output_file_resolved}")
with open(output_file_resolved) as f:
    print(json.dumps(json.load(f), indent=2))

#### (i) Adding a node set definition to a SONATA circuit object

In [ ]:
# Create a neuron set and add it directly to the SONATA circuit object
nset = BiophysicalPopulationPropertyNeuronSet(
    population="S1nonbarrel_neurons",
    property_filter=NeuronPropertyFilter(filter_dict={"layer": ["6"], "synapse_class": ["EXC"]}),
)
nset.set_block_name("L6_EXC")

sonata_circuit = circuit.sonata_circuit
nset_name = nset.add_node_set_definition_to_sonata_circuit(circuit, sonata_circuit)
print(f"Added node set '{nset_name}' to SONATA circuit")
print(f"Node set content: {sonata_circuit.node_sets.content[nset_name]}")
print(f"Node IDs resolved: {nset.get_neuron_ids(circuit)}\n")

# Verify it resolves correctly
resolved_ids = sonata_circuit.nodes['S1nonbarrel_neurons'].ids(nset_name)
print(f"Resolved IDs via SONATA: {resolved_ids}")